# TV3 — Final Test cho LR và RF (CHỈ CHẠY MỘT LẦN)

Chỉ thực hiện sau khi đã chốt siêu tham số và threshold của cả hai mô hình, thống nhất với TV1/TV5. Đây là tập Test cố định đã đóng băng từ TV1.

In [ ]:
# CELL 1 — Cài thư viện (không cài Spark)
%pip install -q "numpy>=1.26,<3" "pandas>=2.1,<3" "pyarrow>=14,<24" "scikit-learn>=1.4,<2" "matplotlib>=3.7,<4" "joblib>=1.3,<2"


In [ ]:
# CELL 2 — Kết nối Drive và chọn đường dẫn (SỬA các biến sau cho phù hợp)
import sys, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Repo: thư mục chứa src/model_lr.py, src/model_rf.py, notebooks/... đã tải từ GitHub
REPO = Path('/content/BDA-Nhom_8-main')
# Drive của TV2: thư mục có train/, val/, test/ và manifest.json
FEATURES_ON_DRIVE = Path('/content/drive/MyDrive/BigData_Topic20/01_data_frozen/features_v1')
# Kết quả lưu trên Drive, tránh mất dữ liệu khi Colab ngắt phiên
OUTPUT_ROOT = Path('/content/drive/MyDrive/BigData_Topic20/02_models/tv3_sklearn')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO / 'src' / 'tv3_common.py').is_file():
    raise FileNotFoundError(f'Thiếu code TV3 tại {REPO}. Hãy upload/clone repo đã ghép file TV3.')
if not (FEATURES_ON_DRIVE / 'train').is_dir():
    raise FileNotFoundError(f'Chưa thấy bảng feature TV2: {FEATURES_ON_DRIVE}')

# Copy dữ liệu sang đĩa cục bộ Colab cho tốc độ đọc tốt hơn.
FEATURES_DIR = Path('/content/features_v1')
if not (FEATURES_DIR / 'train').is_dir():
    shutil.copytree(FEATURES_ON_DRIVE, FEATURES_DIR, dirs_exist_ok=True)
print('REPO:', REPO)
print('FEATURES_DIR:', FEATURES_DIR)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
sys.path.insert(0, str(REPO))


## CELL 3 — Xác nhận file mô hình đã có

In [ ]:
for name in ('lr', 'rf'):
    for filename in ('best_model.joblib','config.json'):
        file = OUTPUT_ROOT/'models'/name/filename
        print(file, 'OK' if file.is_file() else 'MISSING')
        assert file.is_file(), f'Chưa có mô hình cuối {file}' 

## CELL 4 — Chỉ bật khi nhóm đã đồng ý final test. Mỗi mô hình dùng đúng threshold đã khóa trên Validation.

In [ ]:
READY_FOR_FINAL_TEST = False  # CHỈ đổi thành True sau khi đã chốt cả LR và RF
if not READY_FOR_FINAL_TEST:
    print('CHƯA CHẠY TEST: hãy xác nhận final-models với nhóm trước.')
else:
    from src.run_test_tv3 import run_final_test
    final_metrics = run_final_test(FEATURES_DIR, OUTPUT_ROOT)
    print('Đã lưu test_metrics_lr.json và test_metrics_rf.json. Không chạy lại!')

## Đầu ra cuối
`results/test_metrics_lr.json`, `results/test_metrics_rf.json`, `results/tv3_test_comparison.csv`, `predictions_test_*.parquet`. Chia sẻ với TV5 để đưa vào đánh giá chung, báo cáo nhóm và demo.